# 🫀 퀘스트 46 · Q4-J — **오류를 해부하고, 파형을 처음 쓴다**

| | **MedKOS / `notebooks/quest46_q4j_error_anatomy.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 층① 표현, **입력 양식(modality)** 을 바꾼다 |
| 부모 런 | `quest46_q4i_capacity_control`(`20260805T0815`) |
| 성격 | **내 가설이 또 반증됐다. 이번엔 추정기가 아니라 입력이 문제다** |
| 예상 소요 | **20–30분** · **GPU 불필요**(sklearn CPU) |

## ★★★ Q4-I 가 내 「능력 비용」 가설을 반증했다

```
base    9차원  0.9420
shuf   25차원  0.9399     ← 내용 **없는** 16열: 차원 비용은 −0.0021 뿐
both   25차원  0.9278     ← 진짜 16열: −0.0142
C 튜닝 이동 +0.0001 · 기록별 표준화 −0.0077 / −0.0094
`both − shuf` **−0.0121** [−0.0186, −0.0060]
```

차원 비용은 전체 하락의 **15%** 뿐이다. 나머지는 **내용이 실제로 해로운** 것이다.
**추정기는 병목이 아니었다 — 철회한다.**

맞은 건 하나뿐이다: 영점(raw) −0.0241 vs 영점(교정) −0.0680 ⇒ Q4-H 의 −0.0427 은
거의 전부 **교정기 부호 되살림 artifact** 였다. 그건 확인됐지만 **결론은 안 바뀐다**.

### 왜 해로웠나 — **중복**이었다

```
단변량 |AUROC−0.5| — 추가 블록 최고 `pre/base` 0.4209  vs  기존 base 최고 0.4337
DEV 가 56폴드 전부에서 고른 3열 — pre/base(56) · alt_next(56) · alt_prev(53)
```

**`pre/base` 는 base 의 `1 − pre/local_base(k)` 와 사실상 같은 양**이다. 「교과서적
판별자」라며 넣은 게 **이미 있는 조기성 축의 재표현**이었고, 새 정보 없이 가중치만
분산시켜 레코드 간 전이를 나쁘게 했다.

**보상성 휴지기 비율의 실패는 더 근본적이다.** 그건 **PAC(불완전) vs PVC(완전)** 를
가르는 양인데, 우리 과제는 **S vs 나머지**이고 나머지의 대부분은 **정상 N** 이다.
N 은 pause 가 없어 `comp≈1`, PAC 도 `<1` — **둘 다 낮다.** S vs N 을 가르는 양이
아니었다. **문제와 안 맞는 교과서 지식을 넣었다.**

## ★★★ 그래서 진짜 문제 — **입력이 RR 뿐이다**

특징 9개가 **전부 RR 파생**이다. `svdb_data5.npz` 의 **`beat (n, 2, 300)`**
(2리드 · 360Hz 재표본 · R 피크 index 100 · 창 −278~+556ms)는 이 Q4 라인에서
**한 번도 안 썼다.**

| 못 읽는 것 | 임상적 의미 | 상태 |
|---|---|---|
| **이소성 P′** | PAC 의 정의 자체 | Q7-S′ 상한 +0.0213 · 필요 222명(가용 126) → **접음** |
| **QRS 폭·형태** | 넓으면 PVC 또는 **변행전도 SVEB(Ashman)**. RR 만으론 「조기한 N/S/V」를 못 가름 | ★ **미사용** |
| **자기 정상 템플릿 상관** | 그 환자 자신의 정상 QRS 와 얼마나 닮았나 | ★ **미사용** |
| **T 파 안의 P′** | PAC 전형 소견 | 못 봄 |
| **비전도 PAC** | QRS 가 없어 **박동 목록에 없음** | 구조적으로 불가 |
| **AF 의 조기성** | 모든 RR 이 불규칙 → 기준선 붕괴 | ρ(불규칙성, 달성률) −0.6797 |

## 이 런이 하는 것

### ★★★ N1 — 오류 해부 (이 런의 산출물)

**위양성 191개/300개가 무엇인지 한 번도 안 세어봤다.** 실제 주석 심볼(`sym`)로 분해한다.
- 위양성이 대부분 **V(심실조기박동)** 이면 → **형태가 답이다**(QRS 폭이 가름)
- 위양성이 대부분 **N** 이면 → 형태로 못 거른다. **리듬 라우팅·적응증**이 답이다

위음성도 문맥으로 분해한다 — 런/고립 · AF대리 상하위 · 조기성 사분위 · 놓친 점수 백분위.

### ★★★ N2 — 파형 형태 특징 (주 관문)

**레코드 자기 정상 템플릿**(rel-RR ≈ 1 인 박동의 중앙 파형 · **무라벨**) 대비 8열:
QRS 창 상관 · 전체 창 상관 · ST-T 창 상관 · **QRS 폭 대리** · 진폭비 · 면적비 ·
P 창 에너지비. **차원 동일 대조군 `mshuf`** 로 판정한다 —
Q4-I 가 이 방법론의 작동을 확인했다(내용 없는 차원 비용 −0.0021).

### ★★ N3 — 사전등록 **방향**

**SVEB 는 상심실성이라 QRS 가 정상이다.** 그래서 형태는 민감도를 올리는 게 아니라
**위양성을 걸러 PPV 를 올릴** 것이다(현 PPV 0.3620). 사전등록: **ΔPPV@300 > Δ민감도@300**
이고 **V 위양성이 준다**.

### ★ N5 — Q4-I 에서 드러난 영점 방법론 결함

한 rep 안에서 폴드들은 학습 데이터를 90% 공유하므로 **거의 같은 무작위 방향**이 56
레코드에 공통 적용된다. 영점의 유효 표본은 56 이 아니라 **rep 수**다 — 레코드
부트스트랩이 CI 를 **과소평가**한다(raw 영점 `base` 0.5277 이 그 증거).
⇒ rep 을 늘리고 **rep 수준 산포를 병기**한다.

## 관문 (사전등록)

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **N0** | 코호트 · 파형 자산 · Platt 기울기(팔별) | 구성. 깨지면 **중단** |
| **N1 ★★★** | 오류 해부(위양성 심볼 · 위음성 문맥) | 관문 아님 — **이 런의 산출물** |
| **N2 ★★★ 주 관문** | `morph` vs `mshuf` (차원 동일) | 측정된 raw 영점 상단 초과 · 배포·기전 병기 |
| **N3 ★★** | 사전등록 방향 — ΔPPV > Δ민감도 · V 위양성 감소 | 방향 판정 |
| **N4 ★★** | 리듬 라우팅 · 적응증 좁히기 | 관문 아님 |
| **N5 ★** | 영점 rep 수준 산포 | 방법론 |
| **N6** | 필요표본 · 검산표 | R38 ⑦ · R39 ⑤ · R41 ② |

⚠️ **새 데이터 0** — `svdb_data5.npz` 안의 **이미 있는** `beat`·`sym` 을 처음 쓸 뿐이다.
⚠️ **P 갈래 재개가 아니다** — P 창 에너지비는 8열 중 **1열**이고, Q7-S′ 의 정렬·분절
파이프라인을 되살리는 게 아니다.


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(thr)):
        return "⚠️ 미결"
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_pair(a, b, seed, nb=3000, q=2.5):
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 3:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    d = [(b[j] - a[j]).mean() for j in (rng.randint(0, len(a), len(a)) for _ in range(nb))]
    return (float((b - a).mean()), float(np.percentile(d, q)),
            float(np.percentile(d, 100 - q)), len(a))

def need_super(n, half, eff, p80=False):
    if not np.isfinite(half) or not np.isfinite(eff) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260805, 1
RHY_K = (5, 10, 20, 32)
MIN_S, MIN_N = 25, 25
DEV_EVERY = 4
MAIN_K = 300
MAX_NEG_SLOPE, MIN_AUC_SLOPE = 0.10, 0.55
# ── 파형 창(360Hz 재표본 · R 피크 index 100 · 창 −278~+556ms) — **사전 고정**
R_IDX = 100
W_QRS = (85, 125)        # ±69ms — QRS
W_FULL = (60, 220)       # −111 ~ +333ms
W_ST = (130, 260)        # +83 ~ +444ms — ST-T
W_P = (25, 75)           # −208 ~ −69ms — P 자리(PR 120~200ms 와 정합)
W_WID = (80, 130)        # QRS 폭 대리를 재는 창
TMPL_LO, TMPL_HI = 0.92, 1.08     # 템플릿에 넣을 「정상처럼 보이는」 박동 rel-RR 범위
TMPL_MIN = 30                     # 이보다 적으면 레코드 전체 중앙값으로 대체
NB_BOOT = 400 if SMOKE else 2000
N_PERM  = 3   if SMOKE else 10    # ★ Q4-I 결함 수정 — 영점의 유효 표본은 rep 수다

ARMS = ("base", "morph", "mshuf", "monly")
MAIN_CT = "morph-mshuf"
CONTRASTS = (("morph-base",  "base",   "morph"),
             ("morph-mshuf", "mshuf",  "morph"),
             ("monly-base",  "base",   "monly"))
READ_ORDER = ("N0", "N1", "N2", "N3", "N4", "N5", "N6")
SV5 = os.path.join(MITBIH, "svdb_data5.npz")

REF = dict(
    n_ok=56, mean_prev=0.0837,
    q4i=dict(base=0.9420, both=0.9278, shuf=0.9399, base_rz=0.9342, both_rz=0.9184,
             sel=0.9353, base_t=0.9422, both_t=0.9280,
             dim_cost=-0.0021, content=-0.0121, tune_shift=0.0001,
             null_raw=-0.0241, null_cal=-0.0680, null_base_auc=0.5277,
             uni_add=0.4209, uni_base=0.4337,
             sens300=0.7459, ppv300=0.3620, ach=0.8074,
             lo_only=dict(auc=0.9781, ach=0.9533, sens=0.9283),
             rho_irr_ach=-0.6797),
    q7s=dict(ceiling=0.0213, need=222, pool=126),
    lit=[("de Chazal 2004 (IEEE TBME · DS2)", 0.759, 0.385),
         ("Llamedo & Martinez 2011 (IEEE TBME)", 0.77, 0.39)])

RULE_CHECK = {
    "R11 매크로":       "환자 단위 · 상한과 함께 읽는다",
    "R16 fallback 없음": "`beat`·`sym` 없으면 **중단**",
    "R22 누출 없음":     "LORO · 템플릿은 **라벨을 안 쓴다**(rel-RR 로만 고른다)",
    "R26 영점":         "★ 영점은 **raw(비교정)** 으로 · **rep 수준 산포 병기**(Q4-I 결함 수정)",
    "R29 ② 분기 금지":   "N0 이 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":        "관문마다 MDE. **미결 ≠ 등가**",
    "R34 ② 문턱 금지":  "파형 창·템플릿 규칙·예산을 **사전 고정**",
    "R35 ① 자 먼저":    "★★ **N1 오류 해부가 자다** — 위양성이 뭔지 모르면 형태가 답인지 모른다",
    "R38 ⑦ 요약 정합":  "★★★ **「능력 비용」 가설을 철회**한다(Q4-I 실측 차원비용 −0.0021)",
    "R39 ① 대안설명":   "형태가 안 먹히면 **SVEB 는 QRS 가 정상이라서**다 — 그건 기각이 아니라 확인",
    "R40 ① 가시성≠판별": "Q7-S′ 가 P 에서 이미 겪었다 — 형태도 **판별로 전환되는지**만 본다",
    "R41 ② 0 근처":     "효과가 0 근처면 필요표본은 해석 불가",
}

CONFIG = dict(
    exp="quest46_q4j_error_anatomy", quest="ailab-2026-0046", step="error-anatomy",
    parent_exp=["quest46_q4i_capacity_control"],
    purpose=("★★★ **Q4-I 가 내 「능력 비용」 가설을 반증했다 — 철회한다.** 내용 **없는** 16열을 "
             "넣은 `shuf` 가 0.9399 로 base(0.9420)보다 **−0.0021** 밖에 안 떨어졌다. 즉 차원 "
             "비용은 전체 하락 −0.0142 의 **15%** 뿐이고, 나머지 **−0.0121** [−0.0186, −0.0060] "
             "은 **내용이 실제로 해로운** 것이다. `C` 튜닝은 +0.0001(=0), 기록별 표준화는 오히려 "
             "−0.0077/−0.0094 였다. **추정기는 병목이 아니었다.** 맞은 건 하나뿐이다 — 영점(raw) "
             "−0.0241 vs 영점(교정) −0.0680 이라 Q4-H 의 −0.0427 은 거의 전부 **교정기 부호 "
             "되살림 artifact** 였다(확인됐지만 결론은 안 바뀐다). ★★★ **왜 해로웠나 — 중복이다.** "
             "단변량 최고가 추가 블록 `pre/base` 0.4209 vs 기존 base 0.4337 이고, DEV 가 56폴드 "
             "전부에서 고른 3열이 `pre/base`·`alt_next`·`alt_prev` 였다. **`pre/base` 는 base 의 "
             "`1 − pre/local_base(k)` 와 사실상 같은 양**이다. 그리고 **보상성 휴지기 비율의 "
             "실패는 더 근본적**이다 — 그건 **PAC vs PVC** 를 가르는 양인데 우리 과제는 **S vs "
             "나머지**이고 나머지의 대부분은 **정상 N** 이다(N 은 pause 가 없어 comp≈1, PAC 도 "
             "<1 — 둘 다 낮다). **문제와 안 맞는 교과서 지식을 넣었다.** ⇒ 진짜 문제는 **입력이 "
             "RR 뿐**이라는 것이다. 특징 9개가 전부 RR 파생이고, `beat (n,2,300)` 파형은 이 Q4 "
             "라인에서 **한 번도 안 썼다**. ★★★ 그래서 이 런은 ① **오류를 실제 주석 심볼로 "
             "해부**한다(위양성 191/300 이 무엇인지 한 번도 안 세어봤다 — V 면 형태가 답이고 N 이면 "
             "리듬 라우팅이 답이다) ② **레코드 자기 정상 템플릿 대비 형태 8열**을 처음 넣고 "
             "**차원 동일 대조군**으로 판정한다 ③ **사전등록 방향** — SVEB 는 상심실성이라 QRS 가 "
             "정상이므로 형태는 민감도가 아니라 **PPV** 를 올릴 것이다."),
    dataset="SVDB — svdb_data5.npz (새 데이터 0 · GPU 0 · 이미 있는 beat·sym 을 처음 쓴다)",
    arms=list(ARMS), main_contrast=MAIN_CT, main_k=MAIN_K,
    windows=dict(r_idx=R_IDX, qrs=W_QRS, full=W_FULL, st=W_ST, p=W_P, wid=W_WID),
    tmpl=dict(lo=TMPL_LO, hi=TMPL_HI, min_n=TMPL_MIN),
    read_order=READ_ORDER, dev_every=DEV_EVERY, n_boot=NB_BOOT, n_perm=N_PERM,
    smoke=SMOKE, ref=REF, rule_check=RULE_CHECK,
    predictions={
        "N0": "코호트 + 파형 자산 + Platt 기울기(팔별). 깨지면 **중단**",
        "N1": "★★★ **오류 해부(관문 아님 · 이 런의 산출물)** — 예산 300개의 **위양성을 실제 "
              "주석 심볼로** 분해하고, 위음성을 **런/고립 · AF대리 · 조기성 사분위 · 놓친 점수 "
              "백분위**로 분해한다",
        "N2": "★★★ **주 관문** — `morph`(base 9 + 형태 8) vs `mshuf`(차원 동일 · 형태 블록만 "
              "레코드 안에서 공동 행치환). 형태의 **내용**이 신호를 갖는가",
        "N3": "★★ **사전등록 방향** — SVEB 는 QRS 가 정상이므로 형태는 S 를 **더 찾는 게 "
              "아니라** 위양성을 걸러 작동한다. ⚠️ ΔPPV 와 Δ민감도는 예산 고정 시 분자가 "
              "같아(TP) **산술적으로 비교 불가**하므로, 방향은 **위양성 구성**으로 판정한다 — "
              "**심실기원(V/E/F) 감소율 > 상심실정상(N/L/R…) 감소율**",
        "N4": "★★ 리듬 라우팅 — 불규칙 상·하위 별도 모델(전문가 혼합)이 단일 모델보다 나은가 · "
              "규칙 절반만 배포하는 적응증 시나리오",
        "N5": "★ 영점 **rep 수준 산포** 병기(Q4-I 결함 수정)",
        "N6": "필요표본 · 결론 검산표"},
    caveat=("★★★ **「능력 비용」 가설을 철회한다**(R38 ⑦) — Q4-I 의 차원 대조군이 −0.0021 로 "
            "직접 반증했다. 두 런 연속으로 내 가설이 틀렸고, 둘 다 **대조군이 잡았다**. "
            "★★ **형태가 안 먹힐 수도 있고 그건 기각이 아니라 확인이다**(R39 ①) — **SVEB 는 "
            "상심실성이라 QRS 가 정상**이다. 그래서 사전등록 방향을 민감도가 아니라 **PPV** 에 "
            "걸었다. 형태로도 PPV 가 안 오르면 남는 답은 **적응증을 좁히는 것**이고, Q4-I 가 그 "
            "수치를 이미 냈다(규칙 절반만 — AUROC 0.9781 · 달성률 0.9533 · 민감도@300 0.9283). "
            "★★ **P 갈래 재개가 아니다** — P 창 에너지비는 8열 중 1열이고 Q7-S′ 의 정렬·분절 "
            "파이프라인을 되살리는 게 아니다(그 갈래는 상한 +0.0213 · 필요 222명으로 접혔다). "
            "★ **GPU 불필요** — 형태 특징은 numpy 상관·진폭이고 모델은 로지스틱 회귀다. "
            "딥러닝은 이 런에 없다."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q4j_error_anatomy", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q4-J — 오류를 해부하고, 파형을 처음 쓴다**")
run.log(f"  ★★★ **철회** — 「능력 비용」 가설. Q4-I 차원 대조군이 "
        f"{REF['q4i']['dim_cost']:+.4f} 로 반증했고 내용의 값은 {REF['q4i']['content']:+.4f} 였다")
run.log(f"  ★★★ **N1 오류 해부가 이 런의 산출물** — 위양성 {MAIN_K}개 중 "
        f"{int(round(MAIN_K*(1-REF['q4i']['ppv300'])))}개가 무엇인지 한 번도 안 세어봤다")
run.log(f"  ★★★ **N2 주 관문** — 파형 형태 8열 vs **차원 동일 대조군**")
run.log(f"  ★★ **사전등록 방향** — SVEB 는 QRS 가 정상이므로 형태는 **PPV** 를 올린다")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(NB_BOOT={NB_BOOT} · N_PERM={N_PERM})")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【N-0】 코호트 · RR 특징(Q4-I 와 동일 9열) · ★★★ 파형 형태 8열
import pandas as pd
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

run.log("\n" + "=" * 100)
run.log("【N-0】 코호트 · RR 특징은 **그대로** · ★★★ **파형을 처음 쓴다**")
run.log("=" * 100)
VERD, NOTE = {}, {}
def g_(k, v, d):
    VERD[k] = v; NOTE[k] = d; run.log(f"  {k:<5}{v}  {d}")

if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음(R16)")
D5 = np.load(SV5, allow_pickle=True)
for need in ("pid", "y3", "pre_rr", "post_rr", "beat", "sym"):
    if need not in D5.files:
        raise AssetError(f"`{need}` 가 자산에 없다 — 이 런은 **파형과 주석이 있어야** 한다(R16)")
PID = np.asarray(D5["pid"]).astype(int); Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
BEAT_ALL = np.asarray(D5["beat"])
SYM_ALL = np.asarray(D5["sym"]).astype("<U2")
K = np.where(Y3 >= 0)[0]
RID = PID[K]; TT_ = (Y3[K] == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
BEAT = np.asarray(BEAT_ALL[K], dtype="float32"); SYM = SYM_ALL[K]
if BEAT.ndim != 3:
    raise AssetError(f"파형이 (n, 리드, 표본) 이 아니다 — {BEAT.shape}")
NL, LW = BEAT.shape[1], BEAT.shape[2]
if LW <= W_ST[0]:
    raise AssetError(f"파형 길이 {LW} 가 사전 고정 창 {W_ST} 보다 짧다(R34 ②)")
RS = np.array(sorted(set(RID.tolist())))
IDX_ALL = {int(r): np.where(RID == r)[0] for r in RS}
run.log(f"  파형 **{BEAT.shape}** · 리드 {NL} · 길이 {LW} · 주석 심볼 "
        f"{len(set(SYM.tolist()))}종")

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
BASE12 = local_base(12)
REL = pre / (BASE12 + 1e-9)

F_BASE = np.nan_to_num(np.c_[_med - pre,
                             np.column_stack([1.0 - pre / (local_base(k) + 1e-9) for k in RHY_K]),
                             post - pre, np.nan_to_num(_std / (_mean + 1e-9)),
                             np.log1p(np.clip(pre, 0, None)), np.log1p(np.clip(post, 0, None))],
                       nan=0.0, posinf=0.0, neginf=0.0)

# ── ★★★ 파형 형태 특징 — **레코드 자기 정상 템플릿** 대비. 라벨을 안 쓴다(R22).
def _clip(w):
    return (max(0, min(w[0], LW - 2)), max(1, min(w[1], LW)))
WQ, WF, WS, WP, WW = (_clip(w) for w in (W_QRS, W_FULL, W_ST, W_P, W_WID))

def corr_to(x, t):
    """x (m, L, w) · t (L, w) → (m, L) 피어슨 상관"""
    xc = x - x.mean(-1, keepdims=True); tc = t - t.mean(-1, keepdims=True)
    num = (xc * tc).sum(-1)
    den = np.sqrt((xc ** 2).sum(-1) * (tc ** 2).sum(-1)) + 1e-9
    return num / den

N_TMPL_FALLBACK = 0
def morph_feats():
    global N_TMPL_FALLBACK
    out = np.zeros((len(K), 8), float)
    for r in RS:
        ii = IDX_ALL[int(r)]
        okm = (REL[ii] >= TMPL_LO) & (REL[ii] <= TMPL_HI)
        if int(okm.sum()) < TMPL_MIN:
            okm = np.ones(len(ii), bool); N_TMPL_FALLBACK += 1
        B = BEAT[ii]
        T = np.median(B[okm], axis=0)                       # (리드, 길이) — 무라벨 템플릿
        cq = corr_to(B[:, :, WQ[0]:WQ[1]], T[:, WQ[0]:WQ[1]])
        cf = corr_to(B[:, :, WF[0]:WF[1]], T[:, WF[0]:WF[1]])
        cs = corr_to(B[:, :, WS[0]:WS[1]], T[:, WS[0]:WS[1]])
        seg = B[:, :, WW[0]:WW[1]]
        med = np.median(seg, axis=-1, keepdims=True)
        amp = np.abs(seg - med).max(-1, keepdims=True) + 1e-9
        wid = (np.abs(seg - med) > 0.5 * amp).mean(-1)      # QRS 폭 대리(0~1)
        q = B[:, :, WQ[0]:WQ[1]]
        ptp = q.max(-1) - q.min(-1)
        tq = T[:, WQ[0]:WQ[1]]; tptp = float(np.mean(tq.max(-1) - tq.min(-1))) + 1e-9
        area = np.abs(q - np.median(q, axis=-1, keepdims=True)).sum(-1)
        tarea = float(np.mean(np.abs(tq - np.median(tq, axis=-1, keepdims=True)).sum(-1))) + 1e-9
        p = B[:, :, WP[0]:WP[1]]
        pe = np.sqrt((( p - p.mean(-1, keepdims=True)) ** 2).mean(-1))
        tp = T[:, WP[0]:WP[1]]
        tpe = float(np.mean(np.sqrt(((tp - tp.mean(-1, keepdims=True)) ** 2).mean(-1)))) + 1e-9
        out[ii, 0] = cq.min(1); out[ii, 1] = cq.mean(1)
        out[ii, 2] = cf.min(1); out[ii, 3] = cs.min(1)
        out[ii, 4] = wid.mean(1)
        out[ii, 5] = (ptp.mean(1) / tptp)
        out[ii, 6] = (area.mean(1) / tarea)
        out[ii, 7] = (pe.mean(1) / tpe)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

MORPH = morph_feats()
MNAMES = ["corr_qrs_min", "corr_qrs_mean", "corr_full_min", "corr_st_min",
          "qrs_width", "amp_ratio", "area_ratio", "p_energy_ratio"]
run.log(f"  ★★★ 형태 8열 — " + " · ".join(MNAMES))
run.log(f"    템플릿 = 레코드별 **rel-RR ∈ [{TMPL_LO}, {TMPL_HI}] 박동의 중앙 파형** "
        f"(**라벨 안 씀** · 대체 사용 레코드 {N_TMPL_FALLBACK}/{len(RS)})")

_rs = np.random.RandomState(SEED0 + 7)
M_SH = MORPH.copy()
for r in RS:
    ii = IDX_ALL[int(r)]
    M_SH[ii] = MORPH[ii][_rs.permutation(len(ii))]
_moved = float(np.mean(np.any(np.abs(M_SH - MORPH) > 1e-12, axis=1)))
if _moved < 0.5:
    raise AssetError(f"차원 대조군이 거의 항등이다({_moved:.3f}) — 대조군 무효(R35 ①)")

FEAT = {"base": F_BASE, "morph": np.c_[F_BASE, MORPH],
        "mshuf": np.c_[F_BASE, M_SH], "monly": MORPH}
if FEAT["morph"].shape[1] != FEAT["mshuf"].shape[1]:
    raise AssetError("차원 대조군의 차원이 다르다 — 통제 실패")
run.log(f"  차원 — " + " · ".join(f"{a} {FEAT[a].shape[1]}" for a in ARMS)
        + f" · 대조군 이동 {_moved:.1%}")

IDXS = {int(r): IDX_ALL[int(r)] for r in RS}
REC_OK = [int(r) for r in RS
          if TT_[IDXS[int(r)]].sum() >= MIN_S and (~TT_[IDXS[int(r)]]).sum() >= MIN_N]
BURD = {r: float(TT_[IDXS[r]].mean()) for r in REC_OK}
NS_ = {r: int(TT_[IDXS[r]].sum()) for r in REC_OK}
NRE = len(REC_OK); MEAN_PREV = float(np.mean([BURD[r] for r in REC_OK]))
run.log(f"  레코드 {len(RS)} · 채점 가능 **{NRE}** · 평균 유병률 {MEAN_PREV:.4f}")

SLOPES, SLOPE_BY, _CUR = [], {}, [None]
def make_cal(s, y):
    lr = LogisticRegression(max_iter=3000, C=1e6).fit(np.asarray(s).reshape(-1, 1),
                                                       np.asarray(y).astype(int))
    a, b = float(lr.coef_[0, 0]), float(lr.intercept_[0])
    SLOPES.append(a)
    if _CUR[0] is not None: SLOPE_BY.setdefault(_CUR[0], []).append(a)
    return lambda v: a * np.asarray(v, float) + b

def split_rest(held):
    rest = sorted([r for r in REC_OK if r != held], key=lambda r: (BURD[r], r))
    dv = [r for i, r in enumerate(rest) if i % DEV_EVERY == 0]
    return [r for r in rest if r not in set(dv)], dv

def fit_fold(X, held, y_override):
    tr_r, dv_r = split_rest(held)
    tr = np.concatenate([IDXS[r] for r in tr_r]); dv = np.concatenate([IDXS[r] for r in dv_r])
    te = IDXS[held]
    Ftr = X[tr]; fmu = Ftr.mean(0); fsd = Ftr.std(0) + 1e-9
    ytr = TT_[tr].astype(int) if y_override is None else np.asarray(y_override[held], int)
    lr = LogisticRegression(max_iter=3000, C=1.0).fit((Ftr - fmu) / fsd, ytr)
    f = lambda ii: lr.decision_function((X[ii] - fmu) / fsd)
    cl = make_cal(f(dv), TT_[dv]); raw = f(te)
    return te, cl(raw), raw

def loro(X, y_override=None, tag=None, recs=None):
    out = np.full(len(K), np.nan); raw = np.full(len(K), np.nan); _CUR[0] = tag
    for held in (recs if recs is not None else REC_OK):
        te, v, rv = fit_fold(X, held, y_override)
        out[te] = v; raw[te] = rv
    return out, raw

def per_auc(L, recs=None):
    return {r: float(roc_auc_score(TT_[IDXS[r]].astype(int), L[IDXS[r]]))
            for r in (recs if recs is not None else REC_OK)}
def per_ap(L):
    return {r: float(average_precision_score(TT_[IDXS[r]].astype(int), L[IDXS[r]]))
            for r in REC_OK}
def at_k(L, r, k):
    idx = IDXS[r]; sc = L[idx]; yy = TT_[idx]
    k = int(min(max(1, k), len(idx)))
    thr = np.partition(sc, -k)[-k]; fl = sc >= thr
    tp = int((fl & yy).sum()); ceil = min(1.0, k / max(1, NS_[r]))
    return dict(sens=tp / max(1, NS_[r]), ppv=tp / max(1, int(fl.sum())), ceil=ceil,
                ach=(tp / max(1, NS_[r])) / ceil if ceil > 0 else np.nan, flag=fl)
CONFIG["cohort"] = dict(n_ok=NRE, mean_prev=MEAN_PREV, moved=_moved, n_lead=int(NL),
                        beat_len=int(LW), tmpl_fallback=int(N_TMPL_FALLBACK),
                        dims={a: int(FEAT[a].shape[1]) for a in ARMS})
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【N-A】 실행 · N0 · ★★★ N1 오류 해부
run.log("\n" + "=" * 100)
run.log("【N-A】 실행 · N0 · ★★★ **N1 — 위양성이 무엇인지 처음 세어본다**")
run.log("=" * 100)
T0 = time.time()
L, LRAW = {}, {}
for a in ARMS:
    L[a], LRAW[a] = loro(FEAT[a], None, tag=a)
    run.log(f"  ({time.time()-T0:>5.0f}초) {a} 완료")
AUC = {a: per_auc(L[a]) for a in ARMS}
AUC_RAW = {a: per_auc(LRAW[a]) for a in ARMS}
AP = {a: per_ap(L[a]) for a in ARMS}
R300 = {a: {r: at_k(L[a], r, MAIN_K) for r in REC_OK} for a in ARMS}

CAL_GAP = max(abs(AUC[a][r] - AUC_RAW[a][r]) for a in ARMS for r in REC_OK)
run.log(f"\n  ★ 교정 전후 레코드별 AUROC 최대차 **{CAL_GAP:.2e}**(0 이면 단조 증가)")
sl = np.array(SLOPES, float); bad_arms, skipped = [], []
for a in ARMS:
    sa = np.array(SLOPE_BY.get(a, []), float); mac = float(np.mean(list(AUC[a].values())))
    if len(sa) == 0: continue
    negf = float(np.mean(sa <= 0))
    if mac <= MIN_AUC_SLOPE:
        skipped.append((a, mac)); continue
    if np.median(sa) <= 0 or negf > MAX_NEG_SLOPE:
        bad_arms.append((a, float(np.median(sa)), negf))
    run.log(f"    기울기 `{a:<7}` 중앙 {np.median(sa):+.4f} · 음수 {negf:.1%} · AUROC {mac:.4f}")
for a, mac in skipped:
    run.log(f"    ⚠️ `{a}` 는 AUROC {mac:.4f} ≤ {MIN_AUC_SLOPE} — 기울기 부호 검사 건너뜀")
if bad_arms:
    raise AssetError("N0 실패 — 체계적 반전: "
                     + " · ".join(f"{a} {m:+.4f}/{f:.1%}" for a, m, f in bad_arms) + "(R29 ②)")
g_("N0", "✅ 지지", f"파형 {BEAT.shape} · 팔별 기울기 통과 {len(ARMS)-len(skipped)}/{len(ARMS)} · "
                   f"대조군 이동 {_moved:.1%} · 템플릿 대체 {N_TMPL_FALLBACK}")

run.log(f"\n  {'팔':<8}{'차원':>6}{'매크로 AUROC':>15}{'매크로 PR-AUC':>16}"
        f"{'민감도@300':>13}{'PPV@300':>10}{'달성률':>10}")
for a in ARMS:
    run.log(f"  {a:<8}{FEAT[a].shape[1]:>6}{np.mean(list(AUC[a].values())):>15.4f}"
            f"{np.mean(list(AP[a].values())):>16.4f}"
            f"{np.mean([R300[a][r]['sens'] for r in REC_OK]):>13.4f}"
            f"{np.mean([R300[a][r]['ppv'] for r in REC_OK]):>10.4f}"
            f"{np.mean([R300[a][r]['ach'] for r in REC_OK]):>10.4f}")
run.log(f"  (Q4-I 앵커 — base {REF['q4i']['base']} · 민감도@300 {REF['q4i']['sens300']} · "
        f"PPV {REF['q4i']['ppv300']} · 달성률 {REF['q4i']['ach']})")

# ── ★★★ N1 오류 해부
IRR = {r: float(np.sqrt(np.mean(np.diff(pre[IDXS[r]]) ** 2)) / (np.median(pre[IDXS[r]]) + 1e-9))
       for r in REC_OK}
CUT = float(np.median([IRR[r] for r in REC_OK]))
HI = [r for r in REC_OK if IRR[r] >= CUT]; LO = [r for r in REC_OK if IRR[r] < CUT]

def in_run(r):
    """직전 또는 직후 박동도 S 인가(런/이단맥 문맥)"""
    yy = TT_[IDXS[r]]
    pv = np.r_[False, yy[:-1]]; nx = np.r_[yy[1:], False]
    return pv | nx

ANAT = {}
for a in ("base", "morph"):
    fp = Counter(); fp_rec = Counter(); tot_fp = 0
    fn_run = fn_iso = fn_hi = fn_lo = 0; fn_pct = []; fn_rel = []
    tp_tot = 0
    for r in REC_OK:
        idx = IDXS[r]; fl = R300[a][r]["flag"]; yy = TT_[idx]
        bad = fl & (~yy)
        tot_fp += int(bad.sum()); tp_tot += int((fl & yy).sum())
        for s_ in SYM[idx][bad]:
            fp[str(s_)] += 1
        fp_rec[r] = int(bad.sum())
        miss = (~fl) & yy
        if miss.any():
            rr_ = in_run(r)
            fn_run += int((miss & rr_).sum()); fn_iso += int((miss & ~rr_).sum())
            if r in HI: fn_hi += int(miss.sum())
            else: fn_lo += int(miss.sum())
            sc = L[a][idx]
            pctl = np.argsort(np.argsort(sc)) / max(1, len(sc) - 1)   # 레코드 내 점수 백분위
            fn_pct.extend(np.asarray(pctl)[miss].tolist())
            fn_rel.extend(REL[idx][miss].tolist())
    ANAT[a] = dict(fp=dict(fp.most_common()), n_fp=tot_fp, n_tp=tp_tot,
                   fn_run=fn_run, fn_iso=fn_iso, fn_hi=fn_hi, fn_lo=fn_lo,
                   fn_pct_med=float(np.median(fn_pct)) if fn_pct else float("nan"),
                   fn_rel_med=float(np.median(fn_rel)) if fn_rel else float("nan"))

a0 = "base"
A = ANAT[a0]
run.log(f"\n  ★★★ **N1(a) 위양성 해부** — `{a0}` · 예산 {MAIN_K}/기록 · 위양성 **{A['n_fp']}개** "
        f"(적중 {A['n_tp']})")
run.log(f"    {'주석 심볼':<12}{'개수':>9}{'비중':>9}")
for s_, c_ in list(A["fp"].items())[:10]:
    run.log(f"    {s_:<12}{c_:>9}{c_/max(1,A['n_fp']):>9.1%}")
v_share = sum(c for s_, c in A["fp"].items() if s_ in ("V", "E", "F")) / max(1, A["n_fp"])
n_share = sum(c for s_, c in A["fp"].items() if s_ in ("N", "L", "R", "e", "j", "n")) \
    / max(1, A["n_fp"])
run.log(f"    ⇒ **심실기원(V/E/F) {v_share:.1%}** · **상심실 정상(N/L/R/e/j) {n_share:.1%}**")
run.log(f"    ▸ 심실기원이 크면 **QRS 폭·형태가 답**이고, 정상이 크면 형태로는 못 거른다 — "
        f"**리듬 라우팅·적응증**이 답이다")

run.log(f"\n  ★★★ **N1(b) 위음성 해부** — 놓친 S")
run.log(f"    런/이단맥 문맥 **{A['fn_run']}** vs 고립 **{A['fn_iso']}** "
        f"({A['fn_run']/max(1,A['fn_run']+A['fn_iso']):.1%} 가 런)")
run.log(f"    불규칙(AF 대리) 상위 **{A['fn_hi']}** vs 하위 **{A['fn_lo']}** "
        f"({A['fn_hi']/max(1,A['fn_hi']+A['fn_lo']):.1%} 가 불규칙 쪽)")
run.log(f"    놓친 S 의 **레코드 내 점수 백분위 중앙 {A['fn_pct_med']:.3f}** · "
        f"상대 RR 중앙 **{A['fn_rel_med']:.3f}**")
run.log(f"    ▸ 백분위가 높으면 **아깝게 놓친 것**(예산 문제) · 낮으면 **모델이 못 본 것**")
run.log(f"    ▸ 상대 RR 이 1.0 에 가까우면 **조기성 자체가 없다** — RR 로는 원리적으로 못 본다")
g_("N1", "(관문 아님)",
   f"위양성 {A['n_fp']}개 — 심실기원 {v_share:.1%} · 상심실정상 {n_share:.1%} · "
   f"위음성 런 {A['fn_run']/max(1,A['fn_run']+A['fn_iso']):.1%} · 놓친 S 상대RR 중앙 "
   f"{A['fn_rel_med']:.3f} · 점수 백분위 중앙 {A['fn_pct_med']:.3f}")
CONFIG["N0"] = dict(cal_gap=float(CAL_GAP), moved=_moved,
                    macro_auc={a: float(np.mean(list(AUC[a].values()))) for a in ARMS},
                    sens300={a: float(np.mean([R300[a][r]["sens"] for r in REC_OK]))
                             for a in ARMS},
                    ppv300={a: float(np.mean([R300[a][r]["ppv"] for r in REC_OK]))
                            for a in ARMS},
                    ach={a: float(np.mean([R300[a][r]["ach"] for r in REC_OK])) for a in ARMS})
CONFIG["N1"] = dict(anat={a: {k_: v_ for k_, v_ in ANAT[a].items()} for a in ANAT},
                    v_share=float(v_share), n_share=float(n_share), irr_cut=CUT,
                    n_hi=len(HI), n_lo=len(LO))
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【N-B】 ★★★ N2 주 관문 · ★ N5 영점 rep 산포
run.log("\n" + "=" * 100)
run.log("【N-B】 ★★★ N2 — 형태의 **내용**이 신호를 갖는가(차원 동일 대조) · N5 영점 방법론")
run.log("=" * 100)
run.log(f"  ★ 영점은 **raw(비교정)** 으로 읽고, **rep 수준 산포를 병기**한다 — Q4-I 에서 한 rep "
        f"안의 폴드들이 학습 데이터를 90% 공유해 **거의 같은 무작위 방향**이 {NRE} 레코드에 "
        f"공통 적용된다는 게 드러났다(raw 영점 base {REF['q4i']['null_base_auc']:.4f} ≠ 0.5)")
NUL = {c[0]: {r: [] for r in REC_OK} for c in CONTRASTS}
REPM = {c[0]: [] for c in CONTRASTS}
for s_ in range(N_PERM):
    rr = np.random.RandomState(SEED0 + 400 + s_)
    yov = {}
    for held in REC_OK:
        tr_r, _ = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r])
        yov[held] = TT_[tr].astype(int)[rr.permutation(len(tr))]
    An = {a: per_auc(loro(FEAT[a], yov)[1]) for a in ARMS}
    for nm, x_, y_ in CONTRASTS:
        d = [An[y_][r] - An[x_][r] for r in REC_OK]
        REPM[nm].append(float(np.mean(d)))
        for i_, r in enumerate(REC_OK): NUL[nm][r].append(d[i_])
    run.log(f"    ({time.time()-T0:>5.0f}초) 영점 rep {s_+1}/{N_PERM}")

NSTAT, NREP = {}, {}
for nm, x_, y_ in CONTRASTS:
    NSTAT[nm] = boot_mean([float(np.mean(v)) for v in NUL[nm].values()],
                          SEED0 + 61 + len(nm), NB_BOOT)
    v = np.array(REPM[nm], float)
    sd = float(v.std(ddof=1)) if len(v) > 1 else float("nan")
    se = sd / np.sqrt(max(1, len(v)))
    NREP[nm] = dict(mean=float(v.mean()), sd=sd, lo=float(v.mean() - 1.96 * se),
                    hi=float(v.mean() + 1.96 * se), n=int(len(v)))
run.log(f"\n  {'대비':<15}{'영점(레코드 부트스트랩)':>28}{'영점(rep 수준)':>26}{'폭 비':>8}")
for nm, x_, y_ in CONTRASTS:
    a_, b_ = NSTAT[nm], NREP[nm]
    w1 = a_[2] - a_[1]; w2 = b_["hi"] - b_["lo"]
    run.log(f"  {nm:<15}{a_[0]:>+9.4f} [{a_[1]:+.4f},{a_[2]:+.4f}]"
            f"{b_['mean']:>+9.4f} [{b_['lo']:+.4f},{b_['hi']:+.4f}]{w2/max(1e-9,w1):>8.2f}")
run.log(f"  ▸ rep 수준 폭이 더 넓으면 **레코드 부트스트랩이 영점 불확실성을 과소평가**한 것이다")

def two_verdicts(nm, obs):
    """★ 문턱은 **두 영점 상단 중 보수적인 쪽**을 쓴다(Q4-I 결함 수정)"""
    nhi = max(NSTAT[nm][2], NREP[nm]["hi"]) if np.isfinite(NREP[nm]["hi"]) else NSTAT[nm][2]
    thr_dep = max(0.0, nhi) if np.isfinite(nhi) else float("nan")
    return (decide(obs["lo"], obs["hi"], thr_dep, ">"), thr_dep,
            decide(obs["lo"], obs["hi"], nhi, ">"), nhi)

OBS = {}
run.log(f"\n  {'대비':<15}{'Δ 매크로 AUROC':>26}{'영점':>11}{'배포':>9}{'기전':>9}")
TAB = {}
for nm, x_, y_ in CONTRASTS:
    m_, lo_, hi_, n_ = boot_pair([AUC[x_][r] for r in REC_OK], [AUC[y_][r] for r in REC_OK],
                                 SEED0 + 81 + len(nm), NB_BOOT)
    OBS[nm] = dict(mean=m_, lo=lo_, hi=hi_, n=int(n_), mde=float(mde(lo_, hi_)))
    vd, td, vm, tm = two_verdicts(nm, OBS[nm])
    TAB[nm] = dict(obs=OBS[nm], null_rec=list(NSTAT[nm][:3]), null_rep=NREP[nm],
                   thr_deploy=float(td), thr_mech=float(tm), v_deploy=vd, v_mech=vm)
    run.log(f"  {nm:<15}{m_:>+9.4f} [{lo_:+.4f},{hi_:+.4f}]{NSTAT[nm][0]:>+11.4f}{vd:>9}{vm:>9}")

om = OBS[MAIN_CT]; vd, td, vm, tm = two_verdicts(MAIN_CT, om)
run.log(f"\n  ★★★ **N2 주 관문** — `{MAIN_CT}` 는 두 팔의 차원이 "
        f"{FEAT['morph'].shape[1]} 로 동일하다")
run.log(f"    Δ **{om['mean']:+.4f}** [{om['lo']:+.4f}, {om['hi']:+.4f}] · 문턱 {td:+.4f} · "
        f"MDE {om['mde']:.4f}")
g_("N2", vd,
   f"`morph` vs `mshuf`(차원 동일) Δ {om['mean']:+.4f} [{om['lo']:+.4f}, {om['hi']:+.4f}] · "
   f"배포 {vd} · 기전 {vm} — "
   + ("**파형 형태가 RR 너머 신호를 준다**" if vd.startswith("✅") else
      ("형태의 내용이 무작위 정렬과 다르지 않다 — **SVEB 는 QRS 가 정상이라서**일 수 있다(R39 ①)"
       if vd.startswith("❌") else "가르지 못했다(R33 ①)")))
g_("N5", "(관문 아님)",
   f"rep 수준 CI 폭 / 레코드 부트스트랩 폭 = "
   + " · ".join(f"{nm} {(NREP[nm]['hi']-NREP[nm]['lo'])/max(1e-9,(NSTAT[nm][2]-NSTAT[nm][1])):.2f}"
                for nm, _, _ in CONTRASTS))
CONFIG["N2"] = TAB
CONFIG["N5"] = {nm: dict(rec=list(NSTAT[nm][:3]), rep=NREP[nm]) for nm, _, _ in CONTRASTS}
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【N-C】 ★★ N3 사전등록 방향(PPV) · ★★ N4 리듬 라우팅·적응증
run.log("\n" + "=" * 100)
run.log("【N-C】 ★★ N3 — **SVEB 는 QRS 가 정상**이므로 형태는 민감도가 아니라 **PPV** 를 올린다")
run.log("=" * 100)
ds = boot_pair([R300["base"][r]["sens"] for r in REC_OK],
               [R300["morph"][r]["sens"] for r in REC_OK], SEED0 + 141, NB_BOOT)
dp = boot_pair([R300["base"][r]["ppv"] for r in REC_OK],
               [R300["morph"][r]["ppv"] for r in REC_OK], SEED0 + 142, NB_BOOT)
da = boot_pair([R300["base"][r]["ach"] for r in REC_OK],
               [R300["morph"][r]["ach"] for r in REC_OK], SEED0 + 143, NB_BOOT)
run.log(f"  Δ민감도@300 {ds[0]:+.4f} [{ds[1]:+.4f}, {ds[2]:+.4f}]")
run.log(f"  ΔPPV@300    {dp[0]:+.4f} [{dp[1]:+.4f}, {dp[2]:+.4f}]")
run.log(f"  Δ달성률     {da[0]:+.4f} [{da[1]:+.4f}, {da[2]:+.4f}]")
run.log(f"  ⚠️ **ΔPPV 와 Δ민감도를 그냥 비교하면 안 된다** — 예산 k 가 고정이면 "
        f"PPV = TP/k · 민감도 = TP/S 라 **분자가 같다**. S < k 인 기록에서는 "
        f"**산술적으로 Δ민감도 > ΔPPV** 가 강제된다. 그래서 방향 판정을 **위양성 구성**으로 한다")
VSET, NSET = ("V", "E", "F"), ("N", "L", "R", "e", "j", "n")
def grp(a, keys):
    return sum(c for s_, c in ANAT[a]["fp"].items() if s_ in keys)
vfp_b, vfp_m = grp("base", VSET), grp("morph", VSET)
nfp_b, nfp_m = grp("base", NSET), grp("morph", NSET)
red_v = (vfp_b - vfp_m) / max(1, vfp_b); red_n = (nfp_b - nfp_m) / max(1, nfp_b)
run.log(f"  {'위양성 구성':<16}{'base':>9}{'morph':>9}{'감소율':>10}")
run.log(f"  {'심실기원(V/E/F)':<16}{vfp_b:>9}{vfp_m:>9}{red_v:>10.1%}")
run.log(f"  {'상심실정상(N/L/R…)':<16}{nfp_b:>9}{nfp_m:>9}{red_n:>10.1%}")
run.log(f"  {'전체':<16}{ANAT['base']['n_fp']:>9}{ANAT['morph']['n_fp']:>9}"
        f"{(ANAT['base']['n_fp']-ANAT['morph']['n_fp'])/max(1,ANAT['base']['n_fp']):>10.1%}")
dir_ok = red_v > red_n
run.log(f"  ▸ **사전등록 방향**(심실기원 위양성이 상심실정상보다 **더 많이 준다**): "
        + ("✅ 맞았다" if dir_ok else "❌ 틀렸다") + f" ({red_v:.1%} vs {red_n:.1%})")
run.log(f"  ▸ 맞으면 형태는 **V 걸러내기**로 작동한다는 뜻이고, 틀리면 형태의 이득이 "
        f"(있다면) **다른 경로**다 — 그 경우 무엇을 걸러냈는지 위 표에서 읽는다")
g_("N3", "(방향 판정)",
   f"심실기원 위양성 {vfp_b} → {vfp_m}({red_v:.1%} 감소) vs 상심실정상 {nfp_b} → "
   f"{nfp_m}({red_n:.1%}) · ΔPPV {dp[0]:+.4f} · Δ민감도 {ds[0]:+.4f}(산술적으로 비교 불가) · "
   + ("사전등록 방향 **맞음**" if dir_ok else "사전등록 방향 **틀림**"))

run.log("\n" + "=" * 100)
run.log("【N-D】 ★★ N4 — 리듬 라우팅(전문가 혼합) · 적응증 좁히기")
run.log("=" * 100)
run.log(f"  불규칙성(AF 대리) 중앙 {CUT:.4f} — 상위 {len(HI)} · 하위 {len(LO)}")
# ── 전문가 혼합: 같은 절반 안에서만 학습(held-out 은 여전히 제외 · R22)
def loro_within(X, group, tag):
    out = np.full(len(K), np.nan)
    for held in group:
        rest = sorted([r for r in group if r != held], key=lambda r: (BURD[r], r))
        if len(rest) < 4: continue
        dv_r = [r for i, r in enumerate(rest) if i % DEV_EVERY == 0]
        tr_r = [r for r in rest if r not in set(dv_r)]
        if not tr_r or not dv_r: continue
        tr = np.concatenate([IDXS[r] for r in tr_r]); dv = np.concatenate([IDXS[r] for r in dv_r])
        te = IDXS[held]
        Ftr = X[tr]; fmu = Ftr.mean(0); fsd = Ftr.std(0) + 1e-9
        lr = LogisticRegression(max_iter=3000, C=1.0).fit((Ftr - fmu) / fsd, TT_[tr].astype(int))
        f = lambda ii: lr.decision_function((X[ii] - fmu) / fsd)
        _CUR[0] = tag; cl = make_cal(f(dv), TT_[dv]); out[te] = cl(f(te))
    return out
MOE = np.full(len(K), np.nan)
for grp, nm_ in ((HI, "moe_hi"), (LO, "moe_lo")):
    v = loro_within(FEAT["base"], grp, nm_)
    m_ = np.isfinite(v); MOE[m_] = v[m_]
moe_rec = [r for r in REC_OK if np.all(np.isfinite(MOE[IDXS[r]]))]
if len(moe_rec) >= 10:
    a_moe = per_auc(MOE, moe_rec)
    dm = boot_pair([AUC["base"][r] for r in moe_rec], [a_moe[r] for r in moe_rec],
                   SEED0 + 151, NB_BOOT)
    run.log(f"  전문가 혼합(불규칙 상·하위 **별도 학습**) vs 단일 모델 — Δ 매크로 AUROC "
            f"**{dm[0]:+.4f}** [{dm[1]:+.4f}, {dm[2]:+.4f}] (레코드 {len(moe_rec)})")
else:
    dm = (float("nan"),) * 4
    run.log(f"  ⚠️ 전문가 혼합을 채점할 레코드가 부족하다({len(moe_rec)}) — 안 읽는다(R16)")

run.log(f"\n  ▸ **적응증 시나리오** — 규칙 리듬(AF 대리 하위) 절반만 배포한다면")
run.log(f"    {'팔':<8}{'AUROC 전체→하위':>24}{'달성률 전체→하위':>24}{'민감도@300 전체→하위':>26}")
for a in ARMS:
    run.log(f"    {a:<8}{np.mean([AUC[a][r] for r in REC_OK]):>10.4f} → "
            f"{np.mean([AUC[a][r] for r in LO]):<11.4f}"
            f"{np.mean([R300[a][r]['ach'] for r in REC_OK]):>10.4f} → "
            f"{np.mean([R300[a][r]['ach'] for r in LO]):<11.4f}"
            f"{np.mean([R300[a][r]['sens'] for r in REC_OK]):>12.4f} → "
            f"{np.mean([R300[a][r]['sens'] for r in LO]):<11.4f}")
run.log(f"    (Q4-I 실측 `base` 하위만 — AUROC {REF['q4i']['lo_only']['auc']} · 달성률 "
        f"{REF['q4i']['lo_only']['ach']} · 민감도 {REF['q4i']['lo_only']['sens']})")
g_("N4", "(관문 아님)",
   f"전문가 혼합 Δ {dm[0]:+.4f} · 규칙 절반만이면 `base` 민감도@300 "
   f"{np.mean([R300['base'][r]['sens'] for r in REC_OK]):.4f} → "
   f"{np.mean([R300['base'][r]['sens'] for r in LO]):.4f}")
CONFIG["N3"] = dict(d_sens=list(ds[:3]), d_ppv=list(dp[:3]), d_ach=list(da[:3]),
                    vfp_base=int(vfp_b), vfp_morph=int(vfp_m), red_v=float(red_v),
                    nfp_base=int(nfp_b), nfp_morph=int(nfp_m), red_n=float(red_n),
                    dir_ok=bool(dir_ok))
CONFIG["N4"] = dict(moe=list(dm[:3]) if np.isfinite(dm[0]) else [],
                    n_moe=len(moe_rec), cut=CUT,
                    lo_only={a: dict(auc=float(np.mean([AUC[a][r] for r in LO])),
                                     ach=float(np.mean([R300[a][r]["ach"] for r in LO])),
                                     sens=float(np.mean([R300[a][r]["sens"] for r in LO])))
                             for a in ARMS})
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【N-E】 단변량 순위 · 필요표본 · ★ N6 검산표
run.log("\n" + "=" * 100)
run.log("【N-E】 형태 8열의 단변량 순위 · 필요표본 · N6 검산표")
run.log("=" * 100)
def uni(M, names):
    out = []
    for j in range(M.shape[1]):
        a_ = []
        for r in REC_OK:
            ii = IDXS[r]; yy = TT_[ii].astype(int); v = M[ii, j]
            if 0 < yy.sum() < len(ii) and np.std(v) > 0:
                a_.append(abs(roc_auc_score(yy, v) - 0.5))
        out.append(float(np.mean(a_)) if a_ else 0.0)
    return out
UM = uni(MORPH, MNAMES); UB = uni(F_BASE, [f"base{j}" for j in range(F_BASE.shape[1])])
run.log(f"  {'형태 특징':<18}{'|AUROC−0.5|':>14}")
for j in np.argsort(UM)[::-1]:
    run.log(f"  {MNAMES[j]:<18}{UM[j]:>14.4f}")
run.log(f"  ▸ 기존 RR 최고 **{max(UB):.4f}** · 형태 최고 **{max(UM):.4f}** "
        f"(Q4-H 추가블록 최고 {REF['q4i']['uni_add']:.4f})")

eff = om["mean"] - TAB[MAIN_CT]["thr_deploy"]
n5 = need_super(NRE, om["mde"], eff); n8 = need_super(NRE, om["mde"], eff, True)
bad = (not np.isfinite(eff)) or abs(eff) < om["mde"]
run.log(f"\n  N2 주 관문  효과-문턱 {eff:+.4f} · 반폭 {om['mde']:.4f} · n(50%) {n5:.0f} · "
        f"n(80%) {n8:.0f}  "
        + ("★ **해석 불가**(R41 ②)" if bad else
           ("이미 충분하다" if n8 <= NRE else "표본이 더 필요하다")))

run.log("\n  ★ N6 — **결론 검산표**")
CHECK = [
    dict(claim="★★★ 「능력 비용」 가설을 **철회**한다",
         num=f"Q4-I 차원 대조군 `shuf` {REF['q4i']['shuf']} vs base {REF['q4i']['base']} ⇒ 차원 "
             f"비용 **{REF['q4i']['dim_cost']:+.4f}**(전체 −0.0142 의 15%). 내용의 값 "
             f"{REF['q4i']['content']:+.4f} · C 튜닝 이동 {REF['q4i']['tune_shift']:+.4f}",
         assume="**없음** — Q4-I 실측이다",
         iffalse="★★★ 두 런 연속 내 가설이 틀렸고 **둘 다 대조군이 잡았다** — 대조군을 계속 짓는다"),
    dict(claim="★★★ 교과서 특징이 실패한 이유는 **문제와 안 맞는 지식**이었다",
         num=f"보상성 휴지기 비율은 **PAC vs PVC** 를 가르는 양인데 우리 음성의 대부분은 "
             f"**정상 N**(본 런 실측 위양성 중 상심실정상 {CONFIG['N1']['n_share']:.1%} · "
             f"심실기원 {CONFIG['N1']['v_share']:.1%}) · 추가 블록 최고 `pre/base` "
             f"{REF['q4i']['uni_add']:.4f} 는 base 의 `1−pre/local_base` 와 사실상 같은 양",
         assume="`sym` 이 실제 주석이다(자산 검증됨)",
         iffalse="★★ **위양성이 무엇인지 먼저 세는 게 자다**(R35 ①) — 그걸 모르면 어떤 특징이 "
                 "필요한지도 모른다"),
    dict(claim=f"★★★ N2 — 파형 형태의 내용 값 {om['mean']:+.4f} → {VERD['N2']}",
         num=f"두 팔 모두 {FEAT['morph'].shape[1]}차원 · 형태 8열만 레코드 안에서 공동 행치환"
             f"(행 {_moved:.1%} 이동) · 영점 {NSTAT[MAIN_CT][0]:+.4f}(rep 수준 "
             f"{NREP[MAIN_CT]['mean']:+.4f}) · 단변량 최고 {max(UM):.4f}",
         assume="템플릿은 **rel-RR 로만** 고른다 — 라벨을 안 쓴다(R22)",
         iffalse="★★ ❌ 라도 기각이 아니라 **확인**이다(R39 ①) — **SVEB 는 상심실성이라 QRS 가 "
                 "정상**이다. 그러면 남는 답은 **적응증을 좁히는 것**이다"),
    dict(claim=f"★★ N3 사전등록 방향 — 심실기원 감소율 {CONFIG['N3']['red_v']:.1%} vs "
               f"상심실정상 {CONFIG['N3']['red_n']:.1%} → "
               + ("맞음" if CONFIG["N3"]["dir_ok"] else "틀림"),
         num=f"V/E/F {CONFIG['N3']['vfp_base']} → {CONFIG['N3']['vfp_morph']} · "
             f"N/L/R… {CONFIG['N3']['nfp_base']} → {CONFIG['N3']['nfp_morph']} · "
             f"전체 {ANAT['base']['n_fp']} → {ANAT['morph']['n_fp']}",
         assume="⚠️ **ΔPPV vs Δ민감도 비교는 안 쓴다** — 예산 k 고정이면 PPV=TP/k · 민감도=TP/S "
                "로 **분자가 같아** S<k 인 기록에서 Δ민감도>ΔPPV 가 산술적으로 강제된다"
                f"(참고값 ΔPPV {CONFIG['N3']['d_ppv'][0]:+.4f} · Δ민감도 "
                f"{CONFIG['N3']['d_sens'][0]:+.4f})",
         iffalse="★ 이 산술 편향은 **스모크가 잡아준 것**이다 — 원안은 ΔPPV > Δ민감도 였다"),
    dict(claim="★★ N5 — 영점의 유효 표본은 레코드가 아니라 **rep** 이다",
         num=" · ".join(f"{nm} 폭비 "
                        f"{(NREP[nm]['hi']-NREP[nm]['lo'])/max(1e-9,(NSTAT[nm][2]-NSTAT[nm][1])):.2f}"
                        for nm, _, _ in CONTRASTS),
         assume="한 rep 안의 폴드는 학습 데이터를 90% 공유한다",
         iffalse=f"★ Q4-I 의 raw 영점 `base` {REF['q4i']['null_base_auc']:.4f}(≠0.5)가 그 증거였다 "
                 f"— 이제 **두 영점 상단 중 보수적인 쪽**을 문턱으로 쓴다"),
    dict(claim="★ **적응증을 좁히면 다른 제품이 된다**",
         num=f"본 런 `base` 규칙 절반만 — AUROC "
             f"{CONFIG['N4']['lo_only']['base']['auc']:.4f} · 달성률 "
             f"{CONFIG['N4']['lo_only']['base']['ach']:.4f} · 민감도@300 "
             f"{CONFIG['N4']['lo_only']['base']['sens']:.4f}",
         assume="RMSSD/중앙은 **AF 대리**지 AF 진단이 아니다 — 리듬 라벨이 없다",
         iffalse="★★ 실제 배포에선 **리듬 라벨로** 다시 확인해야 한다 — 대리로 고른 코호트는 "
                 "낙관적일 수 있다"),
]
for i, ck in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{ck['claim']}**")
    run.log(f"      근거   {ck['num']}")
    run.log(f"      가정   {ck['assume']}")
    run.log(f"      틀리면 {ck['iffalse']}")
CONFIG["uni"] = dict(morph={MNAMES[j]: UM[j] for j in range(len(MNAMES))},
                     base_max=float(max(UB)), morph_max=float(max(UM)))
CONFIG["need"] = dict(effect=float(eff), half=float(om["mde"]), sup50=float(n5),
                      sup80=float(n8), uninterpretable=bool(bad))
CONFIG["N6"] = CHECK
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【N-F】 그림 · 요약 · 마무리
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))

items = list(ANAT["base"]["fp"].items())[:7]
ax[0].bar(range(len(items)), [c for _, c in items], color="tab:red")
ax[0].set_xticks(range(len(items))); ax[0].set_xticklabels([s for s, _ in items], fontsize=9)
ax[0].set_ylabel("false positives @300"); ax[0].grid(alpha=.3, axis="y")
ax[0].set_title("N1 : what are the false positives (annotation symbol)", fontsize=9)

nm = [c[0] for c in CONTRASTS]
vv = [OBS[n]["mean"] for n in nm]
lo = [OBS[n]["mean"] - OBS[n]["lo"] for n in nm]; hi = [OBS[n]["hi"] - OBS[n]["mean"] for n in nm]
ax[1].errorbar(vv, np.arange(len(nm)), xerr=[lo, hi], fmt="o", capsize=5, color="tab:blue")
ax[1].scatter([NSTAT[n][0] for n in nm], np.arange(len(nm)), marker="x", s=45,
              color="tab:gray", label="null (raw)")
ax[1].axvline(0, color="k", lw=.9)
ax[1].set_yticks(range(len(nm))); ax[1].set_yticklabels(nm, fontsize=8)
ax[1].set_xlabel("macro AUROC delta"); ax[1].legend(fontsize=7)
ax[1].set_title("N2 : does waveform morphology carry content", fontsize=9)
ax[1].grid(alpha=.3, axis="x")

ax[2].scatter([IRR[r] for r in REC_OK], [R300["base"][r]["sens"] for r in REC_OK],
              s=30, color="tab:orange", label="base")
ax[2].scatter([IRR[r] for r in REC_OK], [R300["morph"][r]["sens"] for r in REC_OK],
              s=22, color="tab:green", marker="^", label="morph")
ax[2].axvline(CUT, ls=":", color="tab:gray", lw=1.2)
ax[2].set_xlabel("irregularity (RMSSD / median RR) ~ AF proxy")
ax[2].set_ylabel("sensitivity @300"); ax[2].legend(fontsize=7); ax[2].grid(alpha=.3)
ax[2].set_title("N4 : narrowing the indication", fontsize=9)
fig.tight_layout()
PNG = run.save_fig("q4j_error_anatomy", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
for g in READ_ORDER[:6]:
    run.log(f"  {g:<5}{VERD.get(g, '(관문 아님)')}")
run.log("")
run.log(f"  ★★★ **철회** — 「능력 비용」 가설(Q4-I 차원 대조군 {REF['q4i']['dim_cost']:+.4f})")
run.log(f"  ★★★ **N1 오류 해부** — 위양성 {ANAT['base']['n_fp']}개 중 심실기원 "
        f"{CONFIG['N1']['v_share']:.1%} · 상심실정상 {CONFIG['N1']['n_share']:.1%} · "
        f"놓친 S 의 상대 RR 중앙 {ANAT['base']['fn_rel_med']:.3f} · 런 문맥 "
        f"{ANAT['base']['fn_run']/max(1,ANAT['base']['fn_run']+ANAT['base']['fn_iso']):.1%}")
run.log(f"  ★★★ **N2 주 관문** — `{MAIN_CT}` {om['mean']:+.4f} "
        f"[{om['lo']:+.4f}, {om['hi']:+.4f}] → {VERD['N2']}")
run.log(f"  ★★ **N3 방향** — ΔPPV {CONFIG['N3']['d_ppv'][0]:+.4f} vs Δ민감도 "
        f"{CONFIG['N3']['d_sens'][0]:+.4f} · 심실기원 위양성 {CONFIG['N3']['vfp_base']} → "
        f"{CONFIG['N3']['vfp_morph']} → "
        + ("사전등록 맞음" if CONFIG["N3"]["dir_ok"] else "사전등록 틀림"))
run.log(f"  ★★ **N4 적응증** — 규칙 절반만이면 `base` 민감도@300 "
        f"{np.mean([R300['base'][r]['sens'] for r in REC_OK]):.4f} → "
        f"{CONFIG['N4']['lo_only']['base']['sens']:.4f} · 달성률 "
        f"{CONFIG['N4']['lo_only']['base']['ach']:.4f}")
run.log(f"  ▸ 매크로 AUROC — " + " · ".join(
    f"{a} {np.mean(list(AUC[a].values())):.4f}" for a in ARMS))
run.log(f"  ▸ 형태 단변량 최고 {max(UM):.4f} vs 기존 RR 최고 {max(UB):.4f}")
run.log(f"  ▸ **GPU 안 썼다** — numpy 상관 + 로지스틱 회귀")

run.finish({
    "exp_id": "quest46_q4j_error_anatomy",
    "metric": "macro_auroc_morph_minus_mshuf",
    "value": float(om["mean"]),
    "passed": bool(ok_("N0") and ok_("N2")),
    "summary": ("Q4-I 가 「능력 비용」 가설을 반증했다 — 내용 없는 16열의 비용이 −0.0021 "
                "뿐이라 하락의 85%는 **내용이 실제로 해로운** 것이었다. 그리고 왜 해로웠는지도 "
                "드러났다: 추가 특징이 **이미 있는 조기성 축의 재표현**이었고(단변량 `pre/base` "
                "0.4209 ≈ base 의 `1−pre/local_base`), 보상성 휴지기 비율은 **PAC vs PVC** 를 "
                "가르는 양인데 우리 음성의 대부분은 **정상 N** 이라 **문제와 안 맞는 교과서 "
                "지식**이었다. ⇒ 진짜 문제는 **입력이 RR 뿐**이다. 이 런은 ① 예산 300개의 "
                "**위양성을 실제 주석 심볼로 해부**하고(심실기원이면 형태가 답, 정상이면 리듬 "
                "라우팅이 답) ② `beat (n,2,300)` 파형에서 **레코드 자기 정상 템플릿 대비 형태 "
                "8열**을 처음 만들어 **차원 동일 대조군**으로 판정하고 ③ **SVEB 는 상심실성이라 "
                "QRS 가 정상**이라는 생리로부터 **ΔPPV > Δ민감도** 를 사전등록했다. 덧붙여 Q4-I "
                "에서 드러난 **영점 방법론 결함**(유효 표본이 레코드가 아니라 rep)을 고쳐 "
                "rep 수준 산포를 병기하고 보수적인 문턱을 쓴다."),
    "verdicts": VERD, "notes": NOTE, "rule_check": RULE_CHECK,
    "cohort": CONFIG.get("cohort", {}), "N0": CONFIG.get("N0", {}),
    "N1": CONFIG.get("N1", {}), "N2": CONFIG.get("N2", {}), "N3": CONFIG.get("N3", {}),
    "N4": CONFIG.get("N4", {}), "N5": CONFIG.get("N5", {}), "uni": CONFIG.get("uni", {}),
    "need": CONFIG.get("need", {}), "N6": CONFIG.get("N6", []), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q4j_error_anatomy.ipynb`")
